In [7]:
# =====================================================================
# IMPORTING REQUIRED LIBRARIES
# =====================================================================

import os
import glob
import pandas as pd

from collections import deque
from collections import Counter

from xgboost import XGBRegressor


# =====================================================================
# CONFIGURATION
# =====================================================================

# Main directory where the trained XGBoost models are stored
MODEL_DIR = r"C:\mit_supercloud_raw"

# Available early-observation windows used for the trained models
MODEL_WINDOWS = [10, 30, 60, 120, 180, 240, 300, 360, 600]


# ---------------------------------------------------------------------
# Features used by the trained XGBoost models
# ---------------------------------------------------------------------

FEATURES = [
    "early_power_draw_W",
    "early_utilization_gpu_pct",
    "early_utilization_memory_pct",
    "early_memory_used_MiB",
    "early_memory_free_MiB",
    "early_temperature_gpu",
    "early_temperature_memory",
]


# ---------------------------------------------------------------------
# Thermal operating tiers
# ---------------------------------------------------------------------

TIER_ORDER = [
    "Low",
    "Medium",
    "High",
    "Critical"
]


# Temperature boundaries:
#
# Low       : < 35°C
# Medium    : 35°C to < 45°C
# High      : 45°C to < 71°C
# Critical  : >= 71°C
#
# The value stored for each tier is the lower boundary of that tier.

TEMP_THRESHOLDS = {
    "Low": 35,
    "Medium": 45,
    "High": 71,
}


# ---------------------------------------------------------------------
# Safety ceiling
# ---------------------------------------------------------------------
#
# If the actual GPU sensor temperature reaches 82°C or above,
# the controller immediately forces Critical cooling.
#
# This safety condition bypasses the normal majority-voting logic.

SAFETY_CEILING_C = 82.0


# ---------------------------------------------------------------------
# Number of recent readings used for thermal-tier majority voting
# ---------------------------------------------------------------------

DEBOUNCE_PERSISTENCE = 3


# ---------------------------------------------------------------------
# PWM lookup table configuration
# ---------------------------------------------------------------------

PWM_DIR = r"C:\datacenter_project"

PWM_LOOKUP_FILE = os.path.join(
    PWM_DIR,
    "pwm_lookup_table.csv"
)

PWM_COLUMN = "pwm_pct_clipped"


# =====================================================================
# PWM LOOKUP TABLE
# =====================================================================

def load_pwm_lookup_table(csv_path: str) -> dict:
    """
    Load the data-derived thermal-region to PWM lookup table.

    The CSV file must contain:
        thermal_region
        pwm_pct_clipped

    The controller uses this table instead of hard-coding PWM
    values for Low, Medium, High and Critical cooling tiers.
    """

    if not os.path.isfile(csv_path):
        raise FileNotFoundError(
            f"PWM lookup table not found: {csv_path}"
        )

    table = pd.read_csv(csv_path)

    required = {
        "thermal_region",
        PWM_COLUMN
    }

    missing = required - set(table.columns)

    if missing:
        raise ValueError(
            f"PWM lookup table is missing columns: {sorted(missing)}"
        )

    # Clean thermal-region names
    table["thermal_region"] = (
        table["thermal_region"]
        .astype(str)
        .str.strip()
    )

    # Convert PWM values to numeric
    table[PWM_COLUMN] = pd.to_numeric(
        table[PWM_COLUMN],
        errors="coerce"
    )

    if table[PWM_COLUMN].isna().any():
        raise ValueError(
            "PWM lookup table contains invalid PWM values."
        )

    # Convert CSV rows into dictionary
    lookup = dict(
        zip(
            table["thermal_region"],
            table[PWM_COLUMN]
        )
    )

    # Make sure all four thermal tiers are available
    required_tiers = [
        "Low",
        "Medium",
        "High",
        "Critical"
    ]

    missing_tiers = [
        tier
        for tier in required_tiers
        if tier not in lookup
    ]

    if missing_tiers:
        raise ValueError(
            f"PWM lookup table is missing thermal tiers: "
            f"{missing_tiers}"
        )

    # Return values in the correct thermal-tier order
    return {
        tier: float(lookup[tier])
        for tier in required_tiers
    }


# Load PWM mapping when the controller starts
TIER_TO_FAN_SPEED_PCT = load_pwm_lookup_table(
    PWM_LOOKUP_FILE
)

print(
    f"Loaded PWM lookup table: {PWM_LOOKUP_FILE}"
)

print(
    f"PWM mapping: {TIER_TO_FAN_SPEED_PCT}"
)


# =====================================================================
# LAYER 1 — MODEL LOADING AND MODEL SELECTION
# =====================================================================

class ModelRegistry:
    """
    ModelRegistry manages the trained XGBoost models.

    Responsibilities:
        1. Locate the trained model for each observation window.
        2. Load the XGBoost models.
        3. Select the most appropriate model based on job elapsed time.
        4. Generate the predicted eventual GPU peak temperature.
    """

    def __init__(self, model_dir, windows):
        """
        Initialize the model registry.

        Parameters
        ----------
        model_dir : str
            Main directory containing the trained models.

        windows : list
            Available model windows in seconds.
        """

        self.models = {}

        # Load each available model
        for w in windows:

            # Expected folder structure:
            #
            # MODEL_DIR/
            #     10s/
            #         XGBOOST/
            #             model.json
            #
            #     30s/
            #         XGBOOST/
            #             model.json
            #
            #     ...
            #
            #     600s/
            #         XGBOOST/
            #             model.json

            folder = os.path.join(
                model_dir,
                f"{w}s",
                "XGBOOST"
            )

            if not os.path.isdir(folder):

                print(
                    f"WARNING: Folder not found for "
                    f"{w}s window: {folder} — skipping."
                )

                continue

            # Find XGBoost JSON model files
            json_files = glob.glob(
                os.path.join(folder, "*.json")
            )

            if not json_files:

                print(
                    f"WARNING: No .json model file found "
                    f"inside {folder} — skipping."
                )

                continue

            # If more than one model exists,
            # currently use the first one.
            if len(json_files) > 1:

                print(
                    f"WARNING: Multiple .json files found "
                    f"in {folder}, using the first: "
                    f"{json_files[0]}"
                )

            model_path = json_files[0]

            # Create empty XGBoost regressor
            model = XGBRegressor()

            # Load trained model
            model.load_model(model_path)

            # Store model using its observation window as key
            self.models[w] = model

            print(
                f"Loaded {w}s model from {model_path}"
            )

        # Stop execution if no models were found
        if not self.models:

            raise RuntimeError(
                "No trained models were found. "
                "Check MODEL_DIR and folder structure."
            )

        # Store available windows in ascending order
        self.available_windows = sorted(
            self.models.keys()
        )

        print(
            f"\nLoaded models for windows: "
            f"{self.available_windows} seconds"
        )


    # -----------------------------------------------------------------
    # MODEL WINDOW SELECTION
    # -----------------------------------------------------------------

    def select_window(self, elapsed_seconds):
        """
        Select the largest trained observation window that is
        less than or equal to the current job elapsed time.

        Example:

            elapsed = 150 seconds

            Available:
            10, 30, 60, 120, 180, ...

            Selected model = 120 seconds

        If the elapsed time is smaller than the smallest
        available window, the smallest available model is used.
        """

        usable = []

        for w in self.available_windows:

            if w <= elapsed_seconds:
                usable.append(w)

        if usable:

            return max(usable)

        else:

            return self.available_windows[0]


    # -----------------------------------------------------------------
    # TEMPERATURE PREDICTION
    # -----------------------------------------------------------------

    def predict(self, elapsed_seconds, feature_row: dict):
        """
        Select the appropriate XGBoost model and predict the
        eventual peak GPU temperature.

        Returns
        -------
        predicted_temp : float
            Predicted eventual GPU peak temperature.

        window : int
            Model observation window used.
        """

        # Select appropriate model window
        window = self.select_window(
            elapsed_seconds
        )

        # Retrieve model
        model = self.models[window]

        # Create one-row DataFrame using the exact feature order
        X = pd.DataFrame(
            [feature_row],
            columns=FEATURES
        )

        # Generate prediction
        predicted_temp = float(
            model.predict(X)[0]
        )

        return predicted_temp, window


# =====================================================================
# LAYER 2 — DECISION LOGIC
# =====================================================================
#
# Layer 2 converts:
#
#     Predicted temperature
#             +
#     Actual sensor temperature
#             ↓
#     Thermal tiers
#             ↓
#     Safety logic
#             ↓
#     Majority-voting debounce
#             ↓
#     Final cooling tier
#
# Layer 3 will then convert the final tier into the actuator PWM.
# =====================================================================


# ---------------------------------------------------------------------
# Convert temperature to thermal tier
# ---------------------------------------------------------------------

def temp_to_tier(temp_c: float) -> str:
    """
    Convert a temperature value into a thermal operating tier.

    Thermal regions:

        < 35°C        -> Low
        35 to <45°C   -> Medium
        45 to <71°C   -> High
        >= 71°C       -> Critical
    """

    if temp_c < TEMP_THRESHOLDS["Low"]:

        return "Low"

    elif temp_c < TEMP_THRESHOLDS["Medium"]:

        return "Medium"

    elif temp_c < TEMP_THRESHOLDS["High"]:

        return "High"

    else:

        return "Critical"


# ---------------------------------------------------------------------
# Convert thermal tier into numerical rank
# ---------------------------------------------------------------------

def tier_rank(tier: str) -> int:
    """
    Return the numerical severity rank of a thermal tier.

        Low      = 0
        Medium   = 1
        High     = 2
        Critical = 3
    """

    return TIER_ORDER.index(tier)


# ---------------------------------------------------------------------
# Select the higher-severity tier
# ---------------------------------------------------------------------

def higher_tier(tier_a: str, tier_b: str) -> str:
    """
    Select the more severe of two thermal tiers.

    Example:

        predicted = High
        sensor    = Critical

        result = Critical

    This ensures that the controller does not ignore a
    more severe actual sensor condition.
    """

    if tier_rank(tier_a) >= tier_rank(tier_b):

        return tier_a

    else:

        return tier_b


# ---------------------------------------------------------------------
# Safety ceiling
# ---------------------------------------------------------------------

def apply_safety_ceiling(
    tier: str,
    current_sensor_temp: float
) -> str:
    """
    Apply the hardware safety ceiling.

    If the actual GPU temperature reaches or exceeds 82°C,
    immediately force the thermal tier to Critical.

    Otherwise, preserve the previously calculated tier.
    """

    if current_sensor_temp >= SAFETY_CEILING_C:

        return "Critical"

    return tier


# =====================================================================
# THERMAL-TIER MAJORITY DEBOUNCER
# =====================================================================

class TierDebouncer:
    """
    Reduce the effect of temporary noisy changes in thermal tier.

    The controller uses a 3-sample majority vote rather than
    requiring three consecutive identical readings.

    Example:

        High -> Low -> High

    High appears in 2 out of 3 readings.

        Final tier = High


    Another example:

        Critical -> High -> Critical

    Critical appears in 2 out of 3 readings.

        Final tier = Critical


    The first reading establishes the initial active tier
    immediately.

    A genuine safety-ceiling event bypasses this voting logic
    and immediately forces Critical cooling.
    """

    def __init__(self, persistence=3):

        self.persistence = persistence

        # Store the most recent thermal tiers
        self.history = deque(
            maxlen=persistence
        )

        # Currently active/trusted thermal tier
        self.active_tier = None


    def update(
        self,
        candidate_tier,
        bypass_debounce=False
    ):
        """
        Update the active thermal tier.

        Parameters
        ----------
        candidate_tier : str
            Current thermal tier generated by Layer 2.

        bypass_debounce : bool
            If True, immediately apply the candidate tier.
            Used for safety-ceiling events.

        Returns
        -------
        active_tier : str
            Final trusted thermal tier.
        """

        # -------------------------------------------------------------
        # SAFETY OVERRIDE
        # -------------------------------------------------------------
        #
        # If the GPU reaches the safety ceiling, do not wait for
        # majority voting. Immediately force the requested tier.
        #

        if bypass_debounce:

            self.active_tier = candidate_tier

            self.history.clear()

            self.history.append(candidate_tier)

            return self.active_tier


        # -------------------------------------------------------------
        # STORE THE NEW THERMAL TIER
        # -------------------------------------------------------------

        self.history.append(candidate_tier)


        # -------------------------------------------------------------
        # FIRST READING
        # -------------------------------------------------------------
        #
        # The first valid reading establishes the initial operating
        # state immediately. The controller should not start at Low
        # simply because the history is not yet full.
        #

        if self.active_tier is None:

            self.active_tier = candidate_tier

            return self.active_tier


        # -------------------------------------------------------------
        # MAJORITY VOTING
        # -------------------------------------------------------------
        #
        # Once three readings are available, determine which thermal
        # tier appears most frequently.
        #
        # Example:
        #
        #     High -> Low -> High
        #
        #     High = 2
        #     Low  = 1
        #
        #     Final = High
        #
        # This prevents a single noisy reading from changing the
        # operating tier.
        #

        if len(self.history) == self.persistence:

            counts = Counter(self.history)

            majority_tier, count = (
                counts.most_common(1)[0]
            )

            # A majority means at least 2 out of 3 readings
            if count >= 2:

                self.active_tier = majority_tier


        # Return the currently active/trusted tier
        return self.active_tier


# =====================================================================
# LAYER 3 — ACTUATOR COMMAND
# =====================================================================

def tier_to_actuator_command(
    tier: str,
    cooling_system_type: str = "air"
) -> dict:
    """
    Convert the final cooling tier into an actuator command.

    The PWM value is obtained from the data-derived PWM lookup
    table created during the thermal-region analysis.

    Currently this implementation supports the air/fan cooling
    actuator.
    """

    # Check that the tier exists in the PWM lookup
    if tier not in TIER_TO_FAN_SPEED_PCT:

        raise ValueError(
            f"Unknown cooling tier: {tier}"
        )


    # Currently only air/fan cooling is implemented
    if cooling_system_type != "air":

        raise ValueError(
            "This controller currently implements the "
            "data-derived PWM lookup for air/fan cooling only."
        )


    # Retrieve PWM from lookup table
    pwm_pct = TIER_TO_FAN_SPEED_PCT[tier]


    # Return actuator command
    return {
        "fan_speed_pct": pwm_pct,
        "pwm_pct": pwm_pct
    }


# =====================================================================
# FULL CONTROL PIPELINE
# =====================================================================

class JobCoolingController:
    """
    Complete supervisory controller for one GPU job.

    Processing flow:

        Early telemetry
              ↓
        XGBoost prediction
              ↓
        Predicted thermal tier
              +
        Actual sensor thermal tier
              ↓
        Higher-severity tier
              ↓
        Safety-ceiling check
              ↓
        3-sample majority voting
              ↓
        Final cooling tier
              ↓
        PWM lookup table
              ↓
        Fan actuator command
    """

    def __init__(
        self,
        job_id,
        registry: ModelRegistry,
        cooling_system_type="air"
    ):
        """
        Initialize the controller for one GPU job.
        """

        self.job_id = job_id

        self.registry = registry

        self.cooling_system_type = (
            cooling_system_type
        )

        # Create independent debounce state for this job
        self.debouncer = TierDebouncer(
            persistence=DEBOUNCE_PERSISTENCE
        )


    # -----------------------------------------------------------------
    # ONE CONTROL DECISION CYCLE
    # -----------------------------------------------------------------

    def step(
        self,
        elapsed_seconds: float,
        feature_row: dict,
        current_sensor_temp: float
    ) -> dict:
        """
        Perform one complete control decision cycle.

        Parameters
        ----------
        elapsed_seconds : float
            Time elapsed since the GPU job started.

        feature_row : dict
            Early telemetry features required by the XGBoost model.

        current_sensor_temp : float
            Current GPU temperature from the live sensor.

        Returns
        -------
        dict
            Complete cooling decision information.
        """


        # -------------------------------------------------------------
        # LAYER 1 — PREDICT EVENTUAL PEAK TEMPERATURE
        # -------------------------------------------------------------

        predicted_temp, window_used = (
            self.registry.predict(
                elapsed_seconds,
                feature_row
            )
        )


        # -------------------------------------------------------------
        # CONVERT PREDICTED TEMPERATURE TO THERMAL TIER
        # -------------------------------------------------------------

        predicted_tier = temp_to_tier(
            predicted_temp
        )


        # -------------------------------------------------------------
        # CONVERT ACTUAL SENSOR TEMPERATURE TO THERMAL TIER
        # -------------------------------------------------------------

        sensor_tier = temp_to_tier(
            current_sensor_temp
        )


        # -------------------------------------------------------------
        # COMBINE PREDICTION AND SENSOR INFORMATION
        # -------------------------------------------------------------
        #
        # The more severe tier is selected.
        #
        # Example:
        #
        # Prediction = High
        # Sensor     = Critical
        #
        # Combined   = Critical
        #

        combined_tier = higher_tier(
            predicted_tier,
            sensor_tier
        )


        # -------------------------------------------------------------
        # CHECK SAFETY CEILING
        # -------------------------------------------------------------
        #
        # This flag is kept separately because the debouncer must
        # bypass majority voting during a genuine safety event.
        #

        is_safety_triggered = (
            current_sensor_temp >= SAFETY_CEILING_C
        )


        # Apply safety ceiling
        safe_tier = apply_safety_ceiling(
            combined_tier,
            current_sensor_temp
        )


        # -------------------------------------------------------------
        # APPLY MAJORITY-VOTING DEBOUNCE
        # -------------------------------------------------------------
        #
        # Normal operation:
        #     3 recent readings are considered.
        #
        #     High -> Low -> High
        #         -> High
        #
        # Safety event:
        #     Immediately force Critical.
        #

        final_tier = self.debouncer.update(
            safe_tier,
            bypass_debounce=is_safety_triggered
        )


        # -------------------------------------------------------------
        # LAYER 3 — CONVERT FINAL TIER TO PWM
        # -------------------------------------------------------------

        command = tier_to_actuator_command(
            final_tier,
            self.cooling_system_type
        )


        # -------------------------------------------------------------
        # RETURN COMPLETE DECISION
        # -------------------------------------------------------------

        return {

            "job_id":
                self.job_id,

            "elapsed_seconds":
                elapsed_seconds,

            "model_window_used":
                window_used,

            "predicted_temp":
                round(predicted_temp, 2),

            "predicted_tier":
                predicted_tier,

            "current_sensor_temp":
                current_sensor_temp,

            "sensor_tier":
                sensor_tier,

            "combined_tier":
                combined_tier,

            "safety_triggered":
                is_safety_triggered,

            "final_tier_after_debounce":
                final_tier,

            "actuator_command":
                command
        }


# =====================================================================
# DEMO / TEST HARNESS
# =====================================================================

if __name__ == "__main__":

    # -------------------------------------------------------------
    # LOAD TRAINED MODELS
    # -------------------------------------------------------------
    #
    # IMPORTANT:
    # Models are stored in MODEL_DIR, not PWM_DIR.
    #

    registry = ModelRegistry(
        MODEL_DIR,
        MODEL_WINDOWS
    )


    # -------------------------------------------------------------
    # CREATE CONTROLLER FOR TEST JOB A
    # -------------------------------------------------------------

    controller1 = JobCoolingController(
        job_id="job_A",
        registry=registry,
        cooling_system_type="air"
    )


    # -------------------------------------------------------------
    # SAMPLE INPUT FEATURES
    # -------------------------------------------------------------

    sample_features_1 = {

        "early_power_draw_W":
            150.0,

        "early_utilization_gpu_pct":
            70.0,

        "early_utilization_memory_pct":
            55.0,

        "early_memory_used_MiB":
            12000.0,

        "early_memory_free_MiB":
            20000.0,

        "early_temperature_gpu":
            45.0,

        "early_temperature_memory":
            42.0
    }


    # =============================================================
    # TEST 1 — NORMAL HIGH-TEMPERATURE OPERATION
    # =============================================================

    print(
        "\n--- TEST 1: normal step, 120s elapsed ---"
    )

    result = controller1.step(
        elapsed_seconds=120,
        feature_row=sample_features_1,
        current_sensor_temp=46.0
    )

    for k, v in result.items():

        print(
            f"  {k}: {v}"
        )


    # =============================================================
    # TEST 2 — SAFETY CEILING
    # =============================================================
    #
    # 84°C >= 82°C
    #
    # Therefore:
    #
    #     safety_triggered = True
    #     final_tier = Critical
    #     PWM = Critical PWM
    #
    # No majority-voting delay is allowed.
    # =============================================================

    print(
        "\n--- TEST 2: sensor spike, should force Critical ---"
    )

    result2 = controller1.step(
        elapsed_seconds=125,
        feature_row=sample_features_1,
        current_sensor_temp=84.0
    )

    for k, v in result2.items():

        print(
            f"  {k}: {v}"
        )


    # =============================================================
    # TEST 3 — MAJORITY-VOTING DEBOUNCE
    # =============================================================
    #
    # Sequence:
    #
    #     50°C  -> High
    #     72°C  -> Critical
    #     50°C  -> High
    #
    # Recent tiers:
    #
    #     High -> Critical -> High
    #
    # High occurs twice.
    #
    # Therefore:
    #
    #     Final = High
    #
    #
    # Then:
    #
    #     72°C -> Critical
    #     72°C -> Critical
    #     72°C -> Critical
    #
    # Critical becomes the majority and final tier.
    #
    # =============================================================

    print(
        "\n--- TEST 3: majority-voting behavior "
        "over noisy readings ---"
    )

    controller3 = JobCoolingController(
        job_id="job_B",
        registry=registry,
        cooling_system_type="air"
    )


    noisy_temps = [
        50.0,
        72.0,
        50.0,
        72.0,
        72.0,
        72.0
    ]


    for i, t in enumerate(noisy_temps):

        r = controller3.step(
            elapsed_seconds=200 + i,
            feature_row=sample_features_1,
            current_sensor_temp=t
        )

        print(
            f"  Check {i+1}: "
            f"sensor={t}°C -> "
            f"sensor_tier={r['sensor_tier']} -> "
            f"final_tier={r['final_tier_after_debounce']} "
            f"-> PWM={r['actuator_command']['pwm_pct']}%"
        )


# =====================================================================
# END OF CONTROLLER
# =====================================================================

Loaded PWM lookup table: C:\datacenter_project\pwm_lookup_table.csv
PWM mapping: {'Low': 30.0, 'Medium': 30.0, 'High': 47.4, 'Critical': 100.0}
Loaded 10s model from C:\mit_supercloud_raw\10s\XGBOOST\model_a_10s_xgboost.json
Loaded 30s model from C:\mit_supercloud_raw\30s\XGBOOST\model_a_30s_xgboost.json
Loaded 60s model from C:\mit_supercloud_raw\60s\XGBOOST\model_a_60s_xgboost.json
Loaded 120s model from C:\mit_supercloud_raw\120s\XGBOOST\model_a_120s_xgboost.json

Loaded models for windows: [10, 30, 60, 120] seconds

--- TEST 1: normal step, 120s elapsed ---
  job_id: job_A
  elapsed_seconds: 120
  model_window_used: 120
  predicted_temp: 69.4
  predicted_tier: High
  current_sensor_temp: 46.0
  sensor_tier: High
  combined_tier: High
  safety_triggered: False
  final_tier_after_debounce: High
  actuator_command: {'fan_speed_pct': 47.4, 'pwm_pct': 47.4}

--- TEST 2: sensor spike, should force Critical ---
  job_id: job_A
  elapsed_seconds: 125
  model_window_used: 120
  predicted_te